# 01 · Data Overview & Data Quality

**Project:** Predicting used-car prices from Craigslist listings (`vehicles.csv`)

This notebook loads the raw data, documents every column, and checks data quality.
It does **not** change the data; every cleaning decision is recorded here and applied in `src/clean.py`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Load the data

The raw Kaggle file is 1.45 GB. Almost all of that is free text and links (`description`, `url`,
`region_url`, `image_url`). Text is not allowed for the capstone and is outside the course, so
`src/make_slim.py` removes only those 4 columns and saves `data/vehicles_slim.csv.gz` (16.6 MB).
No rows or values are changed.

In [2]:
df = pd.read_csv('../data/vehicles_slim.csv.gz', low_memory=False)
print(df.shape)
df.head()

(426880, 22)


,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,county,state,lat,long,posting_date
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN


## 2. Data dictionary

Each row is one used-car listing posted on Craigslist in the US between April and May 2021.

| Column | Type | Description |
| --- | --- | --- |
| id | numeric | Unique listing ID |
| region | categorical | Craigslist region the listing was posted in (404 regions) |
| **price** | numeric | **Asking price in US dollars (target)** |
| year | numeric | Model year of the car |
| manufacturer | categorical | Car make (ford, toyota, ...) |
| model | categorical | Car model, typed by the seller (free text: f-150, camry, ...) |
| condition | categorical | Seller-reported condition (salvage → new) |
| cylinders | categorical | Number of engine cylinders, as text ("6 cylinders") |
| fuel | categorical | Fuel type (gas, diesel, hybrid, electric, other) |
| odometer | numeric | Miles driven |
| title_status | categorical | Legal title status (clean, rebuilt, salvage, lien, missing, parts only) |
| transmission | categorical | automatic, manual, other |
| VIN | categorical | Vehicle Identification Number (optional) |
| drive | categorical | 4wd, fwd, rwd |
| size | categorical | compact, sub-compact, mid-size, full-size |
| type | categorical | Body type (sedan, SUV, pickup, ...) |
| paint_color | categorical | Exterior color |
| county | numeric | Empty in the whole file |
| state | categorical | US state (2-letter code, includes DC) |
| lat, long | numeric | Coordinates of the listing |
| posting_date | date | When the listing was posted |

Removed in `make_slim.py` (text and links): `url`, `region_url`, `image_url`, `description`.

## 3. Data types

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 22 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   region        426880 non-null  str    
 2   price         426880 non-null  int64  
 3   year          425675 non-null  float64
 4   manufacturer  409234 non-null  str    
 5   model         421603 non-null  str    
 6   condition     252776 non-null  str    
 7   cylinders     249202 non-null  str    
 8   fuel          423867 non-null  str    
 9   odometer      422480 non-null  float64
 10  title_status  418638 non-null  str    
 11  transmission  424324 non-null  str    
 12  VIN           265838 non-null  str    
 13  drive         296313 non-null  str    
 14  size          120519 non-null  str    
 15  type          334022 non-null  str    
 16  paint_color   296677 non-null  str    
 17  county        0 non-null       float64
 18  state         4

In [4]:
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(exclude='number').columns.tolist()
print(f'{len(num_cols)} numeric columns:', num_cols)
print(f'{len(cat_cols)} text/categorical columns:', cat_cols)

7 numeric columns: ['id', 'price', 'year', 'odometer', 'county', 'lat', 'long']
15 text/categorical columns: ['region', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel', 'title_status', 'transmission', 'VIN', 'drive', 'size', 'type', 'paint_color', 'state', 'posting_date']


`id`, `county`, `lat` and `long` are stored as numbers but are not measurements of the car.
`cylinders` is a number stored as text, and `posting_date` is a date stored as text.

## 4. Null values

In [5]:
nulls = df.isna().sum().to_frame('nulls')
nulls['share_%'] = (nulls['nulls'] / len(df) * 100).round(1)
nulls.sort_values('nulls', ascending=False)

,nulls,share_%
county,426880,100.0
size,306361,71.8
cylinders,177678,41.6
condition,174104,40.8
VIN,161042,37.7
drive,130567,30.6
paint_color,130203,30.5
type,92858,21.8
manufacturer,17646,4.1
title_status,8242,1.9


- `county` is **100% empty** → drop.
- `size` (71.8%) is above the course's ~30% guideline for dropping a column.
- `cylinders`, `condition`, `VIN`, `drive`, `paint_color` and `type` are 22–42% missing: too many rows to drop,
  so whether to keep them depends on whether the missingness carries information (checked in the EDA).
- Everything else is under 5% missing.

## 5. Duplicates and reposted listings

In [6]:
print('Duplicate rows             :', df.duplicated().sum())
print('Duplicate ids              :', df['id'].duplicated().sum())
print('Duplicates ignoring the id :', df.drop(columns='id').duplicated().sum())

Duplicate rows             : 0
Duplicate ids              : 0


Duplicates ignoring the id : 37


There are almost no exact duplicates, because every post gets a new `id`. But the same car is often posted many times:

In [7]:
vin_counts = df['VIN'].value_counts()
print('Rows with a VIN       :', df['VIN'].notna().sum())
print('Distinct VINs         :', len(vin_counts))
print('Most-posted VINs      :', vin_counts.head(5).to_dict())

same_car = df['VIN'].notna() & df.duplicated(['VIN', 'price', 'odometer', 'year', 'model'])
print('\nRepeat posts (same VIN, price, odometer, year, model):', same_car.sum())

Rows with a VIN       : 265838
Distinct VINs         : 118264
Most-posted VINs      : {'1FMJU1JT1HEA52352': 261, '3C6JR6DT3KG560649': 235, '1FTER1EH1LLA36301': 231, '5TFTX4CN3EX042751': 227, '1GCHTCE37G1186784': 214}

Repeat posts (same VIN, price, odometer, year, model): 140545


In [8]:
# Where do the copies of one car appear?
example = vin_counts.index[0]
df.loc[df['VIN'] == example, ['region', 'state', 'price', 'year', 'manufacturer', 'model', 'odometer']].head(8)

,region,state,price,year,manufacturer,model,odometer
76,auburn,al,29590,2017.0,ford,expedition xlt sport,70760.0
707,birmingham,al,29590,2017.0,ford,expedition xlt sport,70760.0
1935,dothan,al,29590,2017.0,ford,expedition xlt sport,70760.0
2201,florence / muscle shoals,al,29590,2017.0,ford,expedition xlt sport,70760.0
2377,gadsden-anniston,al,29590,2017.0,ford,expedition xlt sport,70760.0
3031,huntsville / decatur,al,29590,2017.0,ford,expedition xlt sport,70760.0
4591,montgomery,al,29590,2017.0,ford,expedition xlt sport,70760.0
4878,tuscaloosa,al,29590,2017.0,ford,expedition xlt sport,70760.0


Dealers post the same car, at the same price and mileage, in several nearby regions. These repeats are a
problem for modeling: if copies of one car are in both the train and the test set, the model is tested on cars
it has already seen, and the score looks better than it really is. They are removed in `src/clean.py`
(keeping the first copy). A second rule catches copies without a VIN: the same price, year, manufacturer, model
and exact odometer reading, in any region (private sellers also post in several neighbouring cities).

## 6. Hidden missing values and impossible values

In [9]:
print('price == 0              :', (df['price'] == 0).sum(), f"({(df['price'] == 0).mean():.1%})")
print('price < $1,000          :', (df['price'] < 1000).sum())
print('price > $150,000        :', (df['price'] > 150_000).sum())
print('Largest prices          :', sorted(df['price'].nlargest(6).tolist()))
print('Most common prices      :', df['price'].value_counts().head(5).to_dict())

price == 0              : 32895 (7.7%)
price < $1,000          : 46315
price > $150,000        : 196
Largest prices          : [1410065407, 3009548743, 3024942282, 3024942282, 3736928711, 3736928711]
Most common prices      : {0: 32895, 6995: 3169, 7995: 3129, 9995: 2867, 8995: 2837}


`price = 0` is the most common price: 7.7% of listings use it as "call for price", and most of the
46,315 prices under $1,000 are placeholders or down-payment adverts rather than real asking prices.
At the top, values in the billions (up to 3,736,928,711) are typos or jokes, not prices.

In [10]:
print('odometer == 0           :', (df['odometer'] == 0).sum())
print('odometer == 1           :', (df['odometer'] == 1).sum())
print('odometer >= 999,999     :', (df['odometer'] >= 999_999).sum())
print('Most common odometers   :', df['odometer'].value_counts().head(5).to_dict())

odometer == 0           : 1965
odometer == 1           : 2246


odometer >= 999,999     : 873
Most common odometers   : {100000.0: 2263, 1.0: 2246, 0.0: 1965, 200000.0: 1728, 150000.0: 1603}


In [11]:
print('year range              :', df['year'].min(), '-', df['year'].max())
print('year < 1990             :', (df['year'] < 1990).sum())
dates = pd.to_datetime(df['posting_date'], utc=True)
print('posting dates (UTC)     :', dates.min(), '->', dates.max())

core = ['year', 'manufacturer', 'model', 'odometer', 'fuel', 'transmission']
print('rows with all core fields empty:', df[core].isna().all(axis=1).sum())

year range              : 1900.0 - 2022.0
year < 1990             : 12383


posting dates (UTC)     : 2021-04-04 07:00:25+00:00 -> 2021-05-05 04:24:09+00:00
rows with all core fields empty: 68


- Odometer readings of 0 and 1 are placeholders; readings near 1,000,000 miles are not believable.
- All listings were posted in **one month** (4 April to 5 May 2021), so there is no time trend to model;
  `posting_date` is only useful to compute the car's age (2021 − `year`).
- Model year 2022 is valid: new model years go on sale in spring of the year before.
- 68 rows are empty listings (only region, price and state are filled).

## 7. Summary statistics

In [12]:
df[['price', 'year', 'odometer', 'lat', 'long']].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
price,426880.0,75199.0,12182282.2,0.0,5900.0,13950.0,26485.8,3.736929e+09
year,425675.0,2011.2,9.5,1900.0,2008.0,2013.0,2017.0,2.022000e+03
odometer,422480.0,98043.3,213881.5,0.0,37704.0,85548.0,133542.5,1.000000e+07
lat,420331.0,38.5,5.8,-84.1,34.6,39.2,42.4,8.240000e+01
long,420331.0,-94.7,18.4,-159.8,-111.9,-88.4,-80.8,1.739000e+02


In [13]:
df[cat_cols].describe().T

,count,unique,top,freq
region,426880,404,columbus,3608
manufacturer,409234,42,ford,70985
model,421603,29667,f-150,8009
condition,252776,6,good,121456
cylinders,249202,8,6 cylinders,94169
fuel,423867,5,gas,356209
title_status,418638,6,clean,405117
transmission,424324,3,automatic,336524
VIN,265838,118264,1FMJU1JT1HEA52352,261
drive,296313,3,4wd,131904


## 8. Unique values per categorical column

In [14]:
for col in ['condition', 'cylinders', 'fuel', 'title_status', 'transmission', 'drive', 'size', 'type', 'paint_color']:
    print(f'{col:13s}', df[col].nunique(), sorted(df[col].dropna().unique()))

for col in ['manufacturer', 'model', 'region', 'state']:
    print(f'{col:13s}', df[col].nunique(), 'levels')

condition     6 ['excellent', 'fair', 'good', 'like new', 'new', 'salvage']
cylinders     8 ['10 cylinders', '12 cylinders', '3 cylinders', '4 cylinders', '5 cylinders', '6 cylinders', '8 cylinders', 'other']
fuel          5 ['diesel', 'electric', 'gas', 'hybrid', 'other']
title_status  6 ['clean', 'lien', 'missing', 'parts only', 'rebuilt', 'salvage']
transmission  3 ['automatic', 'manual', 'other']
drive         3 ['4wd', 'fwd', 'rwd']
size          4 ['compact', 'full-size', 'mid-size', 'sub-compact']
type          13 ['SUV', 'bus', 'convertible', 'coupe', 'hatchback', 'mini-van', 'offroad', 'other', 'pickup', 'sedan', 'truck', 'van', 'wagon']
paint_color   12 ['black', 'blue', 'brown', 'custom', 'green', 'grey', 'orange', 'purple', 'red', 'silver', 'white', 'yellow']
manufacturer  42 levels
model         29667 levels
region        404 levels
state         51 levels


`model` has about 29,700 spellings because sellers type it freely ("f-150", "f150 xlt", "F-150 lariat 4x4", ...).
It cannot be one-hot encoded as it is.

## 9. Findings

- **426,880 rows × 22 columns** (after removing 4 text/link columns): 7 stored as numbers, 15 as text.
- **Target `price`** is continuous → a **regression** problem. It needs cleaning: 7.7% are `0` and a few are in the billions.
- **Reposts:** about 180,000 rows repeat a car already in the file → remove before splitting, to avoid leakage between train and test.
- **Nulls:** `county` 100% (drop), `size` 71.8% (above the 30% guideline), six more columns 22–42%.
- **Placeholders:** odometer 0 / 1 and near 1,000,000; prices 0 to $999 and above $150,000.
- **One month of postings** → age = 2021 − year; no time trend.
- **High-cardinality text:** `model` (≈29,700 values), `region` (404), `state` (51).

All cleaning rules are applied in `src/clean.py` and justified with numbers in `02_eda.ipynb`.